# ⚙️ 02 Data Preprocessing
Prepare the dataset for training.

### 🎯 Goals:
1. Ingest dataset
2. Clean corrupted images
3. Handle class imbalance
4. Apply light augmentations

In [ ]:
from google.colab import drive
import os
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/DL/'
DATASET_PATH = os.path.join(BASE_PATH, 'dataset/')
PROCESSED_PATH = os.path.join(BASE_PATH, 'processed_data/')

if not os.path.exists(PROCESSED_PATH):
    os.makedirs(PROCESSED_PATH)

In [ ]:
import PIL
import numpy as np
import tensorflow as tf
from sklearn.utils import class_weight

# 1. Check if dataset exists
if not os.path.exists(DATASET_PATH) or len(os.listdir(DATASET_PATH)) == 0:
    print("📥 Dataset missing. Downloading sample dataset (Flower Photos)...")
    dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
    data_dir = tf.keras.utils.get_file('flower_photos', origin=dataset_url, untar=True)
    DATASET_PATH = data_dir 
else:
    print("✅ Dataset found in Drive.")

### 🧹 Image Cleaning

In [ ]:
import imghdr

def clean_images(data_dir):
    exts = ['jpeg','jpg', 'bmp', 'png']
    for class_folder in os.listdir(data_dir):
        class_path = os.path.join(data_dir, class_folder)
        if os.path.isdir(class_path):
            for image in os.listdir(class_path):
                image_path = os.path.join(class_path, image)
                try:
                    img = PIL.Image.open(image_path)
                    img.verify()
                    if imghdr.what(image_path) not in exts:
                        os.remove(image_path)
                except:
                    os.remove(image_path)

clean_images(DATASET_PATH)
print("✅ Data cleaning complete.")

### ⚖️ Handle Class Imbalance

In [ ]:
def get_class_weights(data_dir):
    classes = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]
    labels = []
    for idx, cls in enumerate(classes):
        num_images = len(os.listdir(os.path.join(data_dir, cls)))
        labels.extend([idx] * num_images)
    
    weights = class_weight.compute_class_weight('balanced', classes=np.unique(labels), y=labels)
    return dict(enumerate(weights))

weights = get_class_weights(DATASET_PATH)
print(f"✅ Class weights: {weights}")

import json
with open(os.path.join(PROCESSED_PATH, 'preprocessing_config.json'), 'w') as f:
    json.dump({'class_weights': {int(k): v for k, v in weights.items()}, 'dataset_path': DATASET_PATH}, f)